# FlyRank Capstone — Leakage-Aware Content Decline Prioritization

## Research question

**For an SEO/content editor, can prior-window and contextual page signals prioritise a review queue that captures a higher share of currently declining pages than a transparent freshness-and-visibility rule?**

> This is **directional decision support**, not an autonomous refresh decision, a causal claim, or a prediction of Google's algorithm. The target is a current-snapshot decline proxy, so all last-30-day, 90-day, and label-derived fields are excluded from model features.

The full written report is in `work/capstone_report.md`. Run the cells below from top to bottom; the analysis script produces reproducible metrics, a ranked queue, permutation importances, and two charts under `work/outputs/`.

## 1. Data and safety contract

The bundled data has 30,000 pseudonymised pages across 32 pseudonymised clients. `client_id` is used only to hold out whole clients for testing; it is not a feature. `trend_direction` defines the label and is never a feature.

The project uses only the **preceding 30-day counts** and stable/contextual signals as predictors. It intentionally excludes every most-recent-30-day field and every 90-day activity total/rate because those overlap the outcome period.

In [1]:
from pathlib import Path
import subprocess
import sys

def locate_repo() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data/raw/content_refresh_anonymized.csv").exists() and (candidate / "work/scripts/run_capstone.py").exists():
            return candidate
    raise FileNotFoundError("Open this notebook from the repository, then run it again.")

REPO = locate_repo()
print(f"Repository found: {REPO}")

Repository found: C:\Users\Danish Computer\OneDrive\Desktop\flyrankmlprojectclone31_7\flyrankmlproject


## 2. Reproducible analysis

The script uses a client-held-out split, a transparent baseline, a random-forest model with missing-data safeguards, and evaluation at the decision-relevant queue sizes (50 and 100 pages).

In [2]:
subprocess.run(
    [sys.executable, str(REPO / "work/scripts/run_capstone.py")],
    cwd=REPO,
    check=True,
)

CompletedProcess(args=['C:\\Program Files\\Python312\\python.exe', 'C:\\Users\\Danish Computer\\OneDrive\\Desktop\\flyrankmlprojectclone31_7\\flyrankmlproject\\work\\scripts\\run_capstone.py'], returncode=0)

## 3. Evaluation evidence

**Primary metric:** Precision@50 on held-out clients, shown alongside the held-out decline base rate. The baseline and the model are compared on exactly the same pages.

Read the table generated by the fresh run below; do not type estimated values into the report.

In [3]:
from IPython.display import Markdown, display
print((REPO / "work/outputs/capstone_metrics.md").read_text())

# Capstone run metrics

| Measure | Value |
|---|---:|
| Rows | 30,000 |
| Pseudonymized clients | 32 |
| Train / held-out rows | 23,837 / 6,163 |
| Train / held-out clients | 25 / 7 |
| Held-out decline base rate | 0.511 |
| Model ROC-AUC | 0.659 |
| Model average precision | 0.616 |
| Model Precision@50 | 0.540 |
| Baseline Precision@50 | 0.340 |
| Model Precision@100 | 0.460 |
| Baseline Precision@100 | 0.340 |

**Interpretation boundary.** This run prioritizes pages that show an observed decline in the
current snapshot using prior-window and contextual signals. It is directional decision support,
not a causal estimate, a forecast beyond this snapshot, or a claim about Google's algorithm.



## 4. Interpretation and recommendation

Permutation importance reports which *feature columns* most helped distinguish the target on the held-out clients. It is not a causal explanation. The operational output is a review queue; the editor must investigate each page before changing it.

In [4]:
import pandas as pd
importance = pd.read_csv(REPO / "work/outputs/capstone_feature_importance.csv")
queue = pd.read_csv(REPO / "work/outputs/capstone_ranked_queue.csv")

print("Top permutation importances")
display(importance.head(10))
print("Top 20 pages for human review")
display(queue.head(20))

Top permutation importances


,feature,importance_mean_auc_drop,importance_std
0,log_impressions_prev_30d,0.140583,0.004687
1,log_clicks_prev_30d,0.010662,0.000644
2,content_age_days,0.009116,0.003321
3,log_sessions_prev_30d,0.005777,0.001210
4,age_tier,0.000607,0.002521
5,search_volume,0.000209,0.001151
6,content_type,0.000000,0.000000
7,competition,-0.000546,0.000198
8,cpc,-0.000787,0.000418
9,char_count,-0.001021,0.001746


Top 20 pages for human review


,priority_rank,content_id,client_id,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,word_count,content_age_days,days_since_last_update,content_type,main_intent,model_decline_score,baseline_priority_score,reason_codes,suggested_action
0,1,content_ff4370afd49c,client_4e07408562,507,1,1,1516.0,280,104,keyword article,commercial,0.965310,0.812275,visible_prior_month,review_for_refresh
1,2,content_700de55b1459,client_4e07408562,408,0,1,1538.0,280,104,keyword article,commercial,0.963256,0.796049,visible_prior_month,review_for_refresh
2,3,content_9ac61c04930e,client_8527a891e2,230,1,4,1504.0,275,104,keyword article,informational,0.960568,0.758583,visible_prior_month,review_for_refresh
3,4,content_c09e49e87ab2,client_8527a891e2,581,0,1,1565.0,238,92,keyword article,commercial,0.958433,0.785892,visible_prior_month,review_for_refresh
4,5,content_98aa0aecb1d9,client_8527a891e2,1880,0,4,1422.0,275,104,keyword article,informational,0.957276,0.903099,visible_prior_month,review_for_refresh
5,6,content_7766ffacdcfa,client_8527a891e2,171,0,3,1477.0,275,104,keyword article,informational,0.955516,0.739834,visible_prior_month,review_for_refresh
6,7,content_a662ef2af9b4,client_8527a891e2,188,0,1,1466.0,275,104,keyword article,informational,0.955342,0.746284,visible_prior_month,review_for_refresh
7,8,content_1d0963b56227,client_4e07408562,867,1,2,1480.0,280,104,keyword article,transactional,0.955119,0.852734,visible_prior_month,review_for_refresh
8,9,content_35d63627bf3e,client_8527a891e2,335,0,1,1592.0,238,103,keyword article,commercial,0.954019,0.754900,visible_prior_month,review_for_refresh
9,10,content_f73c382ac270,client_8527a891e2,323,0,6,1444.0,275,104,keyword article,informational,0.952803,0.785697,visible_prior_month,review_for_refresh


## 5. Limitations and honest claims

1. The label captures an observed decline **inside the current snapshot**; this starter-data version cannot claim to forecast beyond the snapshot.
2. The ranking identifies pages for human review, not a guaranteed refresh outcome.
3. The model does not establish that an editor action will cause traffic to recover. A future warehouse-based extension should define a forward outcome window and assess the result with a time-aware split.
4. No client-identifying information, raw data, or private query text is included in the notebook or the report.

**Next writing step:** after running the notebook, paste the fresh metrics into `work/capstone_report.md`, explain the top importances in plain language, and add a short false-positive/false-negative review before publishing the report.